## 1. Short-term Memory
保存某一次对话的消息和运行状态

### 1.1 记忆保存在内存

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 创建 Agent, 指定 checkpointer
agent = create_agent(
	model="gpt-4o-mini",
	checkpointer=InMemorySaver()
)

In [7]:
from langchain_core.messages import HumanMessage

# 设定 thread_id 作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

response = agent.invoke(
	{"messages": [HumanMessage(content="Hello, I like dogs.")]},
	config
)

print(response)

{'messages': [HumanMessage(content='Hello, I like dogs.', additional_kwargs={}, response_metadata={}, id='8afd2911-dfa6-4d44-8375-346bca8cc957'), AIMessage(content="Hello! That's great to hear! Dogs are wonderful companions. Do you have a favorite breed or a dog of your own?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 13, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8de5ba728b', 'id': 'chatcmpl-EQVAxLyXf98Ua4TicULUOJs741YyF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c36c-ac67-7d22-9f64-402b95bc421f-0', tool_ca

In [8]:
response = agent.invoke(
	{"messages": [HumanMessage(content="What animal do I like?")]},
	config
)

print(response)

{'messages': [HumanMessage(content='Hello, I like dogs.', additional_kwargs={}, response_metadata={}, id='8afd2911-dfa6-4d44-8375-346bca8cc957'), AIMessage(content="Hello! That's great to hear! Dogs are wonderful companions. Do you have a favorite breed or a dog of your own?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 13, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8de5ba728b', 'id': 'chatcmpl-EQVAxLyXf98Ua4TicULUOJs741YyF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c36c-ac67-7d22-9f64-402b95bc421f-0', tool_ca

### 1.2 Memory 持久存储
https://docs.langchain.com/oss/python/integrations/checkpointers

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 连接 sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

# 创建agent
agent = create_agent(
	model="gpt-4o-mini",
	checkpointer=checkpointer
)

In [16]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "thread_2"}}

response = agent.invoke(
	{"messages": [HumanMessage(content="Hello, I like cats.")]},
	config
)

print(response)

{'messages': [HumanMessage(content='Hello, I like cats.', additional_kwargs={}, response_metadata={}, id='88ed54c9-ae6c-4c6a-90af-5ab1267c0b48'), AIMessage(content='Hello! Cats are wonderful creatures. They can be playful, affectionate, and full of personality. Do you have a cat, or is there a specific breed you like?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 13, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f51aa69871', 'id': 'chatcmpl-EQXeYPfv45mCpg83HKpUvcClUlcLB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0

### 1.3 记忆管理

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化 checkpointer
checkpointer = InMemorySaver()

# 初始化中间件
middleware = SummarizationMiddleware(
	model="gpt-4o-mini",
	trigger=("messages", 3),	# 当消息数量达到3条时触发摘要
	keep=("messages", 1)		# 保留最近1条消息, 其他消息将被摘要
)

# 创建 agent
agent = create_agent(
    model="gpt-4.1-mini",
    middleware=[middleware],
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable": {"thread_id": "thread_3"}}

# 制造长会话历史
agent.invoke({"messages": [HumanMessage(content="Hi, my name is Bob")]}, config)
agent.invoke({"messages": [HumanMessage(content="My favorite color is blue")]}, config)
agent.invoke({"messages": [HumanMessage(content="My favorite animal is a cat")]}, config)

# 测试效果
final_response = agent.invoke({"messages": [HumanMessage(content="Do you remember me?")]}, config)

In [ ]:
for message in final_response["messages"]:
	message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT

The user's primary goal is to introduce themselves and seek assistance.

## SUMMARY

The user has introduced themselves as "Bob," stated that their favorite color is blue, and mentioned that their favorite animal is a cat. The assistant responded positively to the introduction and prompted Bob to specify any particular help he might need.

## ARTIFACTS

None

## NEXT STEPS

Await further inquiries or requests from Bob to provide assistance.
================================ Human Message =================================

Do you remember me?
================================== Ai Message ==================================

Hi Bob! Yes, I remember you mentioned that your favorite color is blue and that you like cats. How can I assist you today?


## 2. Long-term Memory
保存跨对话的用户资料、偏好和知识